# ViCare V2 — Trợ lý hỏi đáp sức khỏe tiếng Việt bằng RAG (AI Challenge 24h)

**Repo GitHub chính thức:** `https://github.com/humman0id/vicare-rag`  
**Môi trường chuẩn để Ban giám khảo tái lập:** **Google Colab miễn phí (GPU Tesla T4 — 15 GB VRAM)** *(chọn `Runtime → Change runtime type → T4 GPU`)*

**Tóm tắt các cải tiến V2 so với Baseline (`ViCare_Bootcamp24h.ipynb`):**
1. **Embedding nâng cấp (`google/embeddinggemma-2`) + Hybrid Retrieval (BM25 + Dense RRF) + Đối chứng công bằng với `intfloat/multilingual-e5-small`:**
   - Selective encoder loading (`vision_config=None, audio_config=None`) chỉ nạp nhánh văn bản **271M** tham số (thay vì 740M đa phương thức).
   - **Tối ưu phần cứng T4 miễn phí:** kiểm tra kiến trúc phần cứng `SM >= 8.0` ([`has_native_bf16()`](https://github.com/humman0id/vicare-rag/blob/main/src/vicare/embeddings.py)) để dùng **`float32` gốc** trên Tesla T4 (SM 7.5) thay vì `bfloat16` giả lập phần mềm vốn chậm ~3–5×, và tự động bật `bfloat16` phần cứng nếu chạy trên L4/A100.
   - **Hybrid BM25 (`rank-bm25`) + Vector RRF** (0 MB VRAM thêm, ~5 ms): kết hợp khớp từ khóa y tế tiếng Việt chính xác (tên vaccine, triệu chứng, thuốc) với ngữ nghĩa của `embeddinggemma-2` để nâng đồng thời `Recall@1/3/5/10` và `MRR@10`, trong khi vẫn giữ điểm cosine gốc cho ngưỡng từ chối `MIN_SCORE`.
   - Giải phóng hoàn toàn VRAM của baseline (`e5_embedder.unload()`) trước khi nạp `embeddinggemma-2` để chạy mượt trong 15 GB VRAM T4 cùng Ollama `qwen2.5:3b`.
2. **Prompt cải tiến, Cưỡng chế & Chuẩn hóa Trích dẫn `[1]`, `[2]`, và Tính tất định (`seed=42`):**
   - Cân bằng lại Quy tắc từ chối (khắc phục lỗi *over-refusal* của model 3B ở V1), kèm ví dụ 1 dòng về định dạng `[1]`.
   - Chuẩn hóa các cách viết trích dẫn dài dòng của model nhỏ (như `(trích dẫn [1] và [2])`, `(tài liệu 1)`) về dạng `[1][2]`, loại bỏ nguồn bịa vượt phạm vi, sửa 1 lượt nếu thiếu trích dẫn, và *fail-closed* nếu vẫn không có căn cứ.
   - Cố định `"seed": 42` cùng `"temperature": 0` cho Ollama để kết quả nhất quán giữa cell benchmark và giao diện Gradio.
3. **Hiệu chỉnh ngưỡng từ chối (`MIN_SCORE`) theo phân bố thực tế + Guardrails cấp cứu nâng cấp:**
   - Tự hiệu chỉnh `MIN_SCORE` theo phân bố *trong kho* vs *ngoài kho* riêng của từng model.
   - Nhận diện cấp cứu không dấu & nhận diện phủ định (ví dụ *"không đau ngực"* không gây báo động giả).
4. **Thực nghiệm Matryoshka (MRL `768d / 512d / 256d / 128d`) có số liệu thực (không encode lại):**
   - Đánh đổi dung lượng RAM vector so với Recall@K trong ~2 giây trực tiếp trên vector đã lập chỉ mục.

> ⚠️ **Giới hạn an toàn:** ViCare chỉ là sản phẩm DEMO học thuật trên ViHealthQA, **không phải** công cụ chẩn đoán, cấp cứu hay tư vấn điều trị. Không nhập thông tin cá nhân nhận dạng người thật.


## 0. Kiểm tra GPU Google Colab (T4 miễn phí) & Cài đặt môi trường


In [ ]:
import os, sys, torch

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability(0)
    native_bf16 = cc[0] >= 8 and torch.cuda.is_bf16_supported()
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name} ({total_gb:.1f} GB VRAM) | Compute Capability: SM {cc[0]}.{cc[1]} | Native HW BF16: {native_bf16}")
    if not native_bf16:
        print("ℹ️ GPU hiện tại (T4 SM 7.5) không có tensor core BF16 phần cứng → dùng float32 gốc để tránh chậm do giả lập BF16.")
else:
    print("⚠️ Chưa bật GPU. Trên Google Colab, chọn Runtime -> Change runtime type -> T4 GPU.")


In [ ]:
# Cài thư viện tương thích Google EmbeddingGemma 2 (sentence-transformers>=6.1, transformers>=5.18),
# rank-bm25 cho Hybrid Retrieval, và gói vicare trực tiếp từ GitHub.
%pip install -q -U \
    "chromadb>=1.0,<2" \
    "rank-bm25>=0.2.2,<1" \
    "sentence-transformers>=6.1,<7" \
    "transformers>=5.18,<6" \
    "datasets>=3,<5" \
    "ollama>=0.5,<1" \
    "gradio>=5,<7" \
    "fastapi>=0.115,<1" \
    "uvicorn>=0.30,<1"

if IN_COLAB:
    !rm -rf /content/vicare-rag
    !git clone -q https://github.com/humman0id/vicare-rag.git /content/vicare-rag
    %pip install -q --no-deps -e /content/vicare-rag
    if "/content/vicare-rag/src" not in sys.path:
        sys.path.insert(0, "/content/vicare-rag/src")

import sentence_transformers, transformers, vicare
print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("✅ Đã nạp gói vicare từ:", vicare.__file__)


### 0.1. Cài đặt & khởi động Ollama server + tải LLM (`qwen2.5:3b`)


In [ ]:
import shutil, subprocess, time, requests

# qwen2.5:3b vừa gọn trên T4 miễn phí (15 GB VRAM) và công bằng với baseline.
LLM_MODEL = "qwen2.5:3b"
OLLAMA_URL = "http://127.0.0.1:11434"

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Hãy cài Ollama từ https://ollama.com rồi chạy lại.")
    print("Cài Ollama cho Google Colab...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    cmd = (
        "curl -fL --retry 3 https://ollama.com/download/ollama-linux-amd64.tar.zst "
        "| tar --zstd -xf - -C /usr/local"
    )
    subprocess.run(["bash", "-o", "pipefail", "-c", cmd], check=True)

def server_ready():
    try:
        return requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok
    except requests.RequestException:
        return False

if not server_ready():
    print("Đang khởi động Ollama server...")
    log = open("/tmp/vicare_ollama.log", "w")
    proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(45):
        if server_ready() or proc.poll() is not None:
            break
        time.sleep(1)

if not server_ready():
    raise RuntimeError("Ollama server chưa sẵn sàng. Xem /tmp/vicare_ollama.log")

subprocess.run(["ollama", "pull", LLM_MODEL], check=True)
print("✅ Ollama server & model sẵn sàng:", LLM_MODEL)


## 1. Tải dữ liệu ViHealthQA & tạo tập `dev` cố định (100 câu, `seed=2026`)


In [ ]:
from vicare.data import load_vihealthqa
from vicare.chunking import build_chunks

corpus, dev, gold = load_vihealthqa(dev_seed=2026, dev_size=100)
chunks = build_chunks(corpus, chunk_size=200, overlap=40)

lens = sorted(len(d["text"].split()) for d in corpus)
print(f"Kho tài liệu: {len(corpus):,} văn bản → {len(chunks):,} chunk (CHUNK_SIZE=200, OVERLAP=40)")
print(f"Độ dài theo từ: trung vị {lens[len(lens)//2]}, tối đa {lens[-1]}")
print(f"Tập dev cố định: {len(dev)} câu")


## 2. Chạy Baseline mặc định (`intfloat/multilingual-e5-small`) để đối chứng công bằng


In [ ]:
from dataclasses import replace
from vicare.config import BASELINE_CONFIG
from vicare.embeddings import ViCareEmbedder
from vicare.retriever import ViCareRetriever
from vicare.rag import ViCareRAG
from vicare.evaluation import (
    evaluate_retrieval,
    evaluate_threshold_calibration,
    evaluate_generation,
    save_json_report,
)

N_CHUNKS = None  # None = toàn bộ 11,227 chunk (đánh giá cuối cùng); đặt 2000 nếu muốn thử nhanh.
N_JUDGE = 10     # Giữ nguyên số câu đánh giá giữa Baseline và Bản cải tiến (có thể nâng lên 20).

baseline_cfg = replace(BASELINE_CONFIG, n_chunks=N_CHUNKS, n_judge=N_JUDGE, llm_model=LLM_MODEL)
e5_embedder = ViCareEmbedder(baseline_cfg)
e5_retriever = ViCareRetriever(baseline_cfg, e5_embedder, chunks)
e5_rag = ViCareRAG(baseline_cfg, e5_retriever)

baseline_retrieval = evaluate_retrieval(e5_retriever, dev)
baseline_calibration = evaluate_threshold_calibration(e5_retriever, dev)
baseline_judge = evaluate_generation(e5_rag, dev, n_judge=N_JUDGE)
e5_telemetry = e5_embedder.telemetry()
e5_indexer_telemetry = e5_retriever.telemetry()

baseline_report = {
    "profile": baseline_cfg.profile_name,
    "settings": {
        "embedding_model": baseline_cfg.embedding_model,
        "chunk_size": baseline_cfg.chunk_size,
        "overlap": baseline_cfg.overlap,
        "n_chunks": baseline_cfg.n_chunks,
        "min_score": baseline_cfg.min_score,
        "llm_model": baseline_cfg.llm_model,
        "use_hybrid_bm25": baseline_cfg.use_hybrid_bm25,
    },
    "embedder_telemetry": e5_telemetry,
    "indexer_telemetry": e5_indexer_telemetry,
    "retrieval": baseline_retrieval,
    "calibration": baseline_calibration,
    "judge": baseline_judge,
}
save_json_report("vicare_baseline.json", baseline_report)

# Giải phóng hoàn toàn VRAM của E5 trước khi nạp EmbeddingGemma 2 trên GPU T4 (15 GB).
e5_embedder.unload()

print("=== BASELINE (multilingual-e5-small) ===")
print("Telemetry:", e5_telemetry)
print("Indexer:", e5_indexer_telemetry)
print("Retrieval:", baseline_retrieval)
print("Judge:", baseline_judge)


## 3. Chạy Bản cải tiến V2 (`google/embeddinggemma-2` + Hybrid BM25 RRF + Guardrails V2)

- Tải `google/embeddinggemma-2` nhánh chỉ văn bản (**271M**); dùng **`float32` gốc** trên T4 (SM 7.5) thay vì BF16 giả lập chậm.
- So sánh trực tiếp **Dense-only** vs **Hybrid (BM25 + EmbeddingGemma 2 RRF)** trên cùng 100 câu `dev`.
- Tự động hiệu chỉnh `MIN_SCORE` trên tập trong kho vs ngoài kho, sau đó chạy V2 Prompt + Chuẩn hóa/Cưỡng chế Trích dẫn + `seed=42`.


In [ ]:
from vicare.config import EMBEDDINGGEMMA2_CONFIG

# 3A. Lập chỉ mục một lần với EmbeddingGemma 2 + Hybrid BM25 RRF
improved_cfg_raw = replace(
    EMBEDDINGGEMMA2_CONFIG,
    n_chunks=N_CHUNKS,
    n_judge=N_JUDGE,
    llm_model=LLM_MODEL,
)
gemma_embedder = ViCareEmbedder(improved_cfg_raw)
gemma_retriever = ViCareRetriever(improved_cfg_raw, gemma_embedder, chunks)

# Đo ablation nhanh: Dense-only vs Hybrid BM25 + EmbeddingGemma 2 RRF trên cùng index
gemma_retriever.config = replace(improved_cfg_raw, use_hybrid_bm25=False)
gemma_dense_only_retrieval = evaluate_retrieval(gemma_retriever, dev)

gemma_retriever.config = improved_cfg_raw
improved_retrieval = evaluate_retrieval(gemma_retriever, dev)

print("Ablation Retrieval (cùng 100 câu dev):")
print("  EmbeddingGemma 2 (Dense-only): ", {k: round(v, 4) for k, v in gemma_dense_only_retrieval.items() if k != "n_questions"})
print("  EmbeddingGemma 2 + BM25 (RRF): ", {k: round(v, 4) for k, v in improved_retrieval.items() if k != "n_questions"})

# 3B. Hiệu chỉnh MIN_SCORE trên phân bố điểm cosine thực tế của EmbeddingGemma 2
improved_calibration = evaluate_threshold_calibration(gemma_retriever, dev)
calibrated_min_score = improved_calibration["recommended_min_score"]
print(f"\n✅ Ngưỡng MIN_SCORE hiệu chỉnh cho EmbeddingGemma 2: {calibrated_min_score:.4f}")
print("Phân bố điểm hiệu chỉnh:", improved_calibration)

improved_cfg = replace(improved_cfg_raw, min_score=calibrated_min_score)
gemma_retriever.config = improved_cfg
gemma_rag = ViCareRAG(improved_cfg, gemma_retriever)
improved_judge = evaluate_generation(gemma_rag, dev, n_judge=N_JUDGE)
gemma_telemetry = gemma_embedder.telemetry()
gemma_indexer_telemetry = gemma_retriever.telemetry()

improved_report = {
    "profile": improved_cfg.profile_name,
    "settings": {
        "embedding_model": improved_cfg.embedding_model,
        "chunk_size": improved_cfg.chunk_size,
        "overlap": improved_cfg.overlap,
        "n_chunks": improved_cfg.n_chunks,
        "min_score": improved_cfg.min_score,
        "llm_model": improved_cfg.llm_model,
        "query_prompt_name": improved_cfg.query_prompt_name,
        "truncate_dim": improved_cfg.truncate_dim,
        "use_hybrid_bm25": improved_cfg.use_hybrid_bm25,
        "dense_weight": improved_cfg.dense_weight,
        "bm25_weight": improved_cfg.bm25_weight,
        "ollama_seed": improved_cfg.ollama_seed,
    },
    "embedder_telemetry": gemma_telemetry,
    "indexer_telemetry": gemma_indexer_telemetry,
    "retrieval_dense_only": gemma_dense_only_retrieval,
    "retrieval": improved_retrieval,
    "calibration": improved_calibration,
    "judge": improved_judge,
}

print("\n=== IMPROVED V2 (google/embeddinggemma-2 + Hybrid BM25 RRF + Guardrails V2) ===")
print("Telemetry:", gemma_telemetry)
print("Indexer:", gemma_indexer_telemetry)
print("Retrieval:", improved_retrieval)
print("Judge:", improved_judge)


## 4. Bảng so sánh Trước / Sau & Phân tích Đánh đổi (Trade-offs & Caveats)


In [ ]:
from vicare.evaluation import format_comparison_table

print(format_comparison_table(baseline_report, improved_report))

print("\n--- SO SÁNH TÀI NGUYÊN & ĐÁNH ĐỔI (HONEST TRADE-OFFS) ---")
b_emb, i_emb = baseline_report["embedder_telemetry"], improved_report["embedder_telemetry"]
b_idx, i_idx = baseline_report["indexer_telemetry"], improved_report["indexer_telemetry"]
b_ret, i_ret = baseline_report["retrieval"], improved_report["retrieval"]

rows = [
    ("Model", b_emb["embedding_model"], i_emb["embedding_model"]),
    ("Retrieval Mode", "Dense-only", "Hybrid BM25 + Dense RRF"),
    ("Params (M)", b_emb["param_count_millions"], i_emb["param_count_millions"]),
    ("Vector Dim", b_emb["embedding_dim"], i_emb["embedding_dim"]),
    ("Dtype", b_emb["dtype"], i_emb["dtype"]),
    ("Vector Index (s)", b_idx["index_time_s"], i_idx["index_time_s"]),
    ("BM25 Index (s)", b_idx.get("bm25_index_time_s", 0.0), i_idx.get("bm25_index_time_s", 0.0)),
    ("Avg Query (ms)", b_ret["avg_query_ms"], i_ret["avg_query_ms"]),
    ("Model VRAM (MB)", b_emb["model_vram_mb"], i_emb["model_vram_mb"]),
    ("MIN_SCORE", baseline_report["settings"]["min_score"], improved_report["settings"]["min_score"]),
]
print(f"{'Hạng mục':<18} {'Baseline (E5-small)':<32} {'Improved V2 (Gemma 2 + BM25)':<32}")
print("-" * 82)
for label, v1, v2 in rows:
    print(f"{label:<18} {str(v1):<32} {str(v2):<32}")

save_json_report("vicare_comparison.json", {"baseline": baseline_report, "improved": improved_report})
print("\n✅ Đã lưu vicare_baseline.json và vicare_comparison.json")


### 4.1. Thực nghiệm Matryoshka (MRL `768d → 512d → 256d → 128d`) — Đánh đổi Dung lượng vs Độ chính xác

**Lưu ý kỹ thuật trung thực:** Rút gọn chiều vector bằng Matryoshka (`truncate_dim`) cắt vector **sau** khi đã chạy xong 24 tầng transformer, nên **không** làm giảm thời gian encode neural; lợi ích thực sự là **giảm tới 3×–6× dung lượng RAM/đĩa lưu trữ vector index** với mức suy giảm Recall rất nhỏ.  
Cell dưới đây trích xuất trực tiếp 11,227 vector `768d` đã lưu trong ChromaDB, cắt chiều + **chuẩn hóa lại L2** (bắt buộc theo model card EmbeddingGemma 2) và đo Recall@K thật trong ~2 giây mà không cần encode lại.


In [ ]:
import numpy as np
from vicare.evaluation import evaluate_matryoshka_ablation, format_matryoshka_table

# Lấy toàn bộ vector 768d đã index trong ChromaDB (không cần chạy lại GPU encoding)
stored = gemma_retriever.collection.get(include=["embeddings", "metadatas"])
doc_embeddings = np.asarray(stored["embeddings"], dtype=np.float32)
doc_ids = [m["doc_id"] for m in stored["metadatas"]]
query_embeddings = np.asarray(
    [gemma_embedder.embed_query(q["question"]) for q in dev], dtype=np.float32
)

mrl_results = evaluate_matryoshka_ablation(
    doc_embeddings=doc_embeddings,
    doc_ids=doc_ids,
    query_embeddings=query_embeddings,
    questions=dev,
    dims=(768, 512, 256, 128),
)
print(format_matryoshka_table(mrl_results))

# Lưu kèm kết quả MRL vào file báo cáo so sánh
save_json_report(
    "vicare_comparison.json",
    {"baseline": baseline_report, "improved": improved_report, "mrl_ablation": mrl_results},
)
print("\n✅ Đã cập nhật kết quả Matryoshka vào vicare_comparison.json")


### 4.2. Minh chứng định tính (Trong kho, Cấp cứu, Phủ định không cấp cứu, Ngoài kho)


In [ ]:
probe_questions = [
    ("TRONG KHO", "Đang chích ngừa viêm gan B có chích ngừa Covid-19 được không?"),
    ("CẤP CỨU", "Trẻ bị sốt cao kèm co giật thì phải làm gì?"),
    ("PHỦ ĐỊNH (không báo động giả)", "Tôi bị ho nhẹ vài ngày, không đau ngực và không khó thở, có cần đi khám không?"),
    ("NGOÀI PHẠM VI", "Thủ đô của nước Úc là gì?"),
]

for tag, q in probe_questions:
    res = gemma_rag.ask(q)
    print(f"=== [{tag}] {q} ===")
    print(res["answer"])
    print(
        f"Refused={res['refused']} ({res['refusal_reason']}) | "
        f"Emergency={res['emergency_triggered']} {res['emergency_categories']} | "
        f"TopScore={res['top_score']} / MinScore={res['min_score']}"
    )
    print(f"CitationAudit={res['citation_audit']}")
    for s in res["sources"]:
        mark = "✓" if s["cited"] else " "
        print(f"  [{s['index']}]{mark} {s['url'] or s['doc_id']} (score={s['score']:.3f})")
    print()


## 5. Demo giao diện Gradio tương tác (hiển thị nguồn trích dẫn & trạng thái điểm)


In [ ]:
ENABLE_GRADIO = True  # Đổi thành False nếu chỉ muốn chạy benchmark.

if ENABLE_GRADIO:
    from vicare.ui import build_demo
    demo = build_demo(gemma_rag)
    demo.launch(share=IN_COLAB, debug=False, show_error=True)
else:
    print("Đã bỏ qua Gradio. Đặt ENABLE_GRADIO = True để mở demo.")


---

## 6. Lưu trữ Hạn chế, Sự cố đã nhận diện & Bài học thực nghiệm (Archival Notes: V1 → V2)

Mục này ghi lại minh bạch toàn bộ hạn chế kỹ thuật đã phát hiện trong quá trình chạy thực tế trên **Google Colab miễn phí (Tesla T4)** từ bản V1 đến V2, cùng các giới hạn cấu trúc còn tồn tại của hệ thống:

### A. Các hạn chế / sự cố đã phát hiện ở V1 và cách xử lý ở V2
1. **Giả lập `bfloat16` phần mềm trên Tesla T4 (SM 7.5) làm chậm lập chỉ mục ~5×:**
   - *Hiện tượng ở V1:* `torch.cuda.is_bf16_supported()` trên PyTorch mới trả về `True` nhờ giả lập phần mềm, khiến `google/embeddinggemma-2` mất **316.4 giây** (so với **63.8 giây** của `e5-small`) và **83.4 ms/truy vấn** (so với **23.7 ms**).
   - *Khắc phục ở V2:* Kiểm tra năng lực phần cứng thực (`SM >= 8.0`, [`has_native_bf16()`](https://github.com/humman0id/vicare-rag/blob/main/src/vicare/embeddings.py)) để T4 chạy `float32` gốc (chỉ ~1.0 GB VRAM trong 14.6 GB VRAM T4, vẫn giữ quy tắc tuyệt đối không dùng `float16`), còn L4/A100 tự động dùng `bfloat16` phần cứng.
2. **`Recall@1` giảm nhẹ ở V1 (`36.0% → 33.0%`) dù `Recall@10` tăng mạnh (`54.0% → 59.0%`):**
   - *Nguyên nhân:* Tìm kiếm vector thuần (dense-only) nhóm các bài tư vấn y tế cùng chủ đề rất sát nhau (`0.780–0.785`), làm giảm độ phân biệt các cụm từ khóa y khoa chính xác (tên vaccine, thuốc, triệu chứng đặc hiệu).
   - *Khắc phục ở V2:* Bổ sung **Hybrid Retrieval (BM25 `rank-bm25` + `embeddinggemma-2` qua Weighted Reciprocal Rank Fusion)** — tốn **0 MB VRAM thêm** và giữ nguyên điểm cosine thực tế để không phá vỡ ngưỡng `MIN_SCORE`.
3. **Từ chối quá mức (*over-refusal*) & rơi `Judge@10` (`70.0% → 60.0%`) ở V1:**
   - *Nguyên nhân 1 (Prompt quá khắt khe cho model 3B):* Quy tắc *"không trả lời trực tiếp câu hỏi"* khiến `qwen2.5:3b` từ chối cả câu hỏi có tài liệu liên quan (`in_scope_refusal` tăng từ `30% → 40%`, và câu kiểm thử phủ định ho nhẹ bị từ chối dù `TopScore = 0.7961`).
   - *Nguyên nhân 2 (Định dạng trích dẫn & tính không tất định):* Ở một số lượt gọi trong Gradio (ví dụ câu cấp cứu *"Trẻ bị sốt cao kèm co giật..."*), `qwen2.5:3b` sinh câu trả lời nhưng bỏ dấu `[1]` hoặc viết dạng văn xuôi `(trích dẫn [1])` / `(tài liệu 1)`, dẫn tới từ chối `missing_valid_citations` dù `TopScore = 0.7901`.
   - *Khắc phục ở V2:* Nới nhẹ quy tắc tổng hợp tài liệu liên quan (để việc chặn ngoài phạm vi cho ngưỡng `MIN_SCORE` đã hiệu chỉnh đạt 100% OOS refusal), thêm ví dụ 1 dòng `[1]` trong prompt & repair prompt, chuẩn hóa trích dẫn văn xuôi tiếng Việt trong [`verify_and_clean_citations()`](https://github.com/humman0id/vicare-rag/blob/main/src/vicare/guardrails.py), cố định `"seed": 42` trong Ollama, và hiển thị trạng thái `top_score / min_score / refusal_reason` ngay trên giao diện Gradio.
4. **Hiệu chỉnh ngưỡng `MIN_SCORE` bắt buộc khi đổi mô hình Embedding:**
   - Thang điểm cosine của `multilingual-e5-small` nằm quanh `0.85–0.93` (ngưỡng gốc `0.86`), còn `google/embeddinggemma-2` nằm quanh `0.57–0.85` (ngưỡng hiệu chỉnh `0.6715`, biên phân tách `+0.0749`). Dùng lại `0.86` sẽ gây từ chối sai gần 100%.
5. **Phụ thuộc thư viện của `google/embeddinggemma-2`:**
   - Yêu cầu `sentence-transformers>=6.1` và `transformers>=5.18` (phiên bản `<6` trong notebook gốc không nạp được `embedding_gemma2`), đồng thời `AutoProcessor` vẫn cần `torchvision` ngay cả khi chỉ nạp nhánh văn bản (`vision_config=None, audio_config=None`).
   - Khi cài đặt trên Colab có thể xuất hiện cảnh báo xung đột phiên bản của các gói có sẵn không liên quan (`tweepy/oauthlib`, `google-adk/opentelemetry`), không ảnh hưởng luồng chạy ViCare.
6. **Hiểu đúng về rút gọn Matryoshka (`truncate_dim=256`):**
   - Ở bản thử trung gian, cell MRL chỉ tạo config mà chưa đo thực tế và nhầm rằng cắt chiều giúp giảm thời gian encode neural. Thực tế MRL giảm dung lượng lưu trữ/RAM index (tới 3×–6×), còn thời gian chạy transformer giữ nguyên; V2 đã thay bằng thực nghiệm định lượng thực tế ở Mục 4.1.

### B. Các giới hạn cấu trúc còn tồn tại (cần lưu ý khi đánh giá & sử dụng)
- **Đánh giá trên kho đã biết (*closed-corpus retrieval*):** `ViHealthQA` dùng chính câu trả lời của tập dữ liệu làm kho kiến thức; `Recall@K` phản ánh khả năng tìm lại đáp án gốc trong kho, chưa thay thế kiểm thử tổng quát hóa trên nguồn y khoa ngoài tập dữ liệu.
- **Kích thước mẫu `Judge@N` và thiên lệch tự chấm:** `N_JUDGE = 10` câu là mẫu nhỏ, và dùng chính `qwen2.5:3b` làm giám khảo (`LLM-as-a-judge`) có thể mang thiên lệch — chỉ mang tính tham khảo định lượng, cần kết hợp kiểm tra thủ công.
- **Giới hạn an toàn lâm sàng:** ViCare **không phải** hệ thống chẩn đoán hay tư vấn điều trị. Dữ liệu `ViHealthQA` (2022) có thể đã lỗi thời với hướng dẫn y tế mới nhất; cơ chế nhận diện từ khóa cấp cứu & ngưỡng similarity chỉ là lớp bảo vệ học thuật, không bảo đảm an toàn lâm sàng tuyệt đối.
- **Giới hạn độ trễ & mô hình 3B:** Trên GPU T4 miễn phí, `qwen2.5:3b` vẫn có giới hạn về suy luận đa bước và tổng hợp nhiều tài liệu dài so với các mô hình lớn hơn (`7B+`), đồng thời tốc độ encode của `embeddinggemma-2` (271M) nặng hơn `multilingual-e5-small` (118M).
